# Sensitivity test result

In [39]:
import xarray as xr

In [40]:
ds = xr.open_dataset(
    "/Users/kns5/Data/level2/calibration_sensitivity/alc_level2_calibration_sensitivity.nc"
)

In [41]:
ds

<xarray.Dataset> Size: 11MB
Dimensions:                           (instrument_type: 6,
                                       drift_pct_per_decade: 61, time: 3650,
                                       sigma_level: 5, bound: 2)
Coordinates:
  * instrument_type                   (instrument_type) <U6 144B 'chm15k' ......
  * drift_pct_per_decade              (drift_pct_per_decade) float32 244B -60...
  * time                              (time) datetime64[ns] 29kB 2010-01-01 ....
  * sigma_level                       (sigma_level) float64 40B 0.1974 ... 0....
    sigma_level_names                 (sigma_level) <U9 180B ...
  * bound                             (bound) <U5 40B 'lower' 'upper'
Data variables:
    low_cloud_fraction_daily_series   (instrument_type, drift_pct_per_decade, time) float32 5MB ...
    low_cloud_fraction_daily_anomaly  (instrument_type, drift_pct_per_decade, time) float32 5MB ...
    low_cloud_fraction_trend          (instrument_type, drift_pct_per_decade) float32 1kB ...
    low_cloud_fraction_trend_pvalue   (instrument_type, drift_pct_per_decade) float32 1kB ...
    low_cloud_fraction_trend_std      (instrument_type, drift_pct_per_decade, sigma_level) float32 7kB ...
    low_cloud_fraction_trend_ci       (instrument_type, drift_pct_per_decade, sigma_level, bound) float32 15kB ...
    n_valid_days                      (instrument_type, drift_pct_per_decade) float32 1kB ...
    drift_fraction_per_decade         (drift_pct_per_decade) float32 244B ...
    start_factor                      (drift_pct_per_decade) float32 244B ...
    midpoint_factor                   (drift_pct_per_decade) float32 244B ...
    end_factor                        (drift_pct_per_decade) float32 244B ...
    scenario_label                    (drift_pct_per_decade) <U19 5kB ...
Attributes:
    title:                           ALC calibration-drift sensitivity experi...
    description:                     Synthetic 10-year ALC series sampled fro...
    synthetic_start_date:            2010-01-01T00:00:00
    synthetic_n_years:               10
    random_seed:                     42
    trend_akritas_confidence_level:  68.27
    level4_reference_variable:       low_cloud_fraction

In [42]:
figwidth = 20 / 2.56 / 2

In [43]:
baseline = ds["low_cloud_fraction_trend"].sel(drift_pct_per_decade=0.0)
baseline_trend_ci = ds["low_cloud_fraction_trend_ci"].sel(
    drift_pct_per_decade=0.0
)

In [52]:
normalized_effect = (ds["low_cloud_fraction_trend"] - baseline) * 10 * 100
normalized_effect.sel(drift_pct_per_decade=slice(-20, 20))
normalized_effect_ci = (
    (ds["low_cloud_fraction_trend_ci"] - baseline) * 10 * 100
)

In [53]:
normalized_effect.drift_pct_per_decade

<xarray.DataArray 'drift_pct_per_decade' (drift_pct_per_decade: 61)> Size: 244B
array([-60.      , -57.966103, -55.932205, -53.898304, -51.864407, -49.83051 ,
       -47.79661 , -45.76271 , -43.728813, -41.694916, -39.66102 , -37.627117,
       -35.59322 , -33.559322, -31.525423, -29.491526, -27.457626, -25.423729,
       -23.38983 , -21.355932, -19.322035, -17.288136, -15.254237, -13.220339,
       -11.18644 ,  -9.152542,  -7.118644,  -5.084746,  -3.050848,  -1.016949,
         0.      ,   1.016949,   3.050848,   5.084746,   7.118644,   9.152542,
        11.18644 ,  13.220339,  15.254237,  17.288136,  19.322035,  21.355932,
        23.38983 ,  25.423729,  27.457626,  29.491526,  31.525423,  33.559322,
        35.59322 ,  37.627117,  39.66102 ,  41.694916,  43.728813,  45.76271 ,
        47.79661 ,  49.83051 ,  51.864407,  53.898304,  55.932205,  57.966103,
        60.      ], dtype=float32)
Coordinates:
  * drift_pct_per_decade  (drift_pct_per_decade) float32 244B -60.0 ... 60.0

In [54]:
import numpy as np

In [59]:
normalized_effect_ci_1sigma = normalized_effect_ci.sel(
    sigma_level=0.6827, method="nearest"
)
std_estimate = normalized_effect_ci_1sigma.sel(
    bound="upper"
) - normalized_effect_ci_1sigma.sel(bound="lower")
print(f"Estimated drift effect size is {np.abs(normalized_effect).min():.2e} to {np.abs(normalized_effect).max():.2e} percentage points per decade.")
print(
    f"Estimated aleatoric uncertainty is {(std_estimate / np.abs(normalized_effect)).min():.2f} to {(std_estimate / np.abs(normalized_effect.where(normalized_effect != 0))).max():.2f} times the estimated drift effect size."
)

Estimated drift effect size is 0.00e+00 to 3.39e-02 percentage points per decade.
Estimated aleatoric uncertainty is 85.14 to 997439.75 times the estimated drift effect size.


<xarray.DataArray ()> Size: 4B
array(85.1444, dtype=float32)
Coordinates:
    sigma_level        float64 8B 0.6827
    sigma_level_names  <U9 36B '1sigma'

In [17]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(figwidth, figwidth * 0.6))

# choose ~1-sigma (68.27%) CI level
ci_sel = normalized_effect_ci.sel(sigma_level=0.6827, method="nearest")
x = normalized_effect["drift_pct_per_decade"].values

for inst in normalized_effect.instrument_type.values:
    if inst == "ld40":
        continue

    y = normalized_effect.sel(instrument_type=inst).values
    ci_inst = ci_sel.sel(instrument_type=inst)
    y_low = ci_inst.sel(bound="lower").values
    y_high = ci_inst.sel(bound="upper").values
    
    print(f"{}")

    (line,) = ax.plot(x, y, label=inst)
    ax.fill_between(x, y_low, y_high, color=line.get_color(), alpha=0.2, linewidth=0)

ax.set_xlabel("Calibration drift (% per decade)")
ax.set_ylabel("Change in $f_{\\mathrm{low}}$ trend (%-pt dec$^{-1}$)")
ax.legend()
ax.axhline(0, color="k", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

SyntaxError: f-string: valid expression required before '}' (2265421273.py, line 18)